<a href="https://colab.research.google.com/github/FunGrape/Labs_ML_Demchyshyn/blob/main/%D0%9B%D0%A09_%D0%94%D0%B5%D0%BC%D1%87%D0%B8%D1%88%D0%B8%D0%BD_4_9_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Демчишин Дмитро Володимирович
**Група:** ФІТ 4-9  
**Дата:** 25.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [ ]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [ ]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [ ]:
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [ ]:
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()
min_rating = ratings_df['rating'].min()
max_rating = ratings_df['rating'].max()
mean_rating = ratings_df['rating'].mean()
missing_values = ratings_df.isnull().sum().sum()

print(f"Кількість унікальних користувачів: {n_users}")
print(f"Кількість унікальних фільмів: {n_items}")
print(f"Оцінки: мін = {min_rating}, макс = {max_rating}, середнє = {mean_rating:.2f}")
print(f"Кількість пропущених значень: {missing_values}")

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Оцінки: мін = 1.0, макс = 5.0, середнє = 3.53
Кількість пропущених значень: 0


**Короткі спостереження щодо даних:**

Набір даних `MovieLens 100K` містить 100 000 оцінок від 943 унікальних користувачів для 1682 фільмів. Розподіл оцінок знаходиться в межах від 1.0 до 5.0 із середнім значенням 3.53, що свідчить про переважання позитивних відгуків серед користувачів. У наборі даних повністю відсутні пропущені значення.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [ ]:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print(f"Розмір train_df: {train_df.shape}")
print(f"Розмір test_df: {test_df.shape}")

Розмір train_df: (80000, 4)
Розмір test_df: (20000, 4)


In [ ]:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = [
    (row['user_id'], row['item_id'], float(row['rating']))
    for _, row in test_df.iterrows()
]

print(f"Кількість записів у testset: {len(testset)}")

Кількість записів у testset: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [ ]:
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

* **Для SVD:**
  * `n_factors`: кількість прихованих факторів (50 та 100) впливає на здатність моделі уловлювати складні залежності у вподобаннях.
  * `lr_all`: швидкість навчання (0.005 та 0.010) визначає розмір кроку градієнтного спуску при оптимізації.
  * `reg_all`: параметр регуляризації (0.02 та 0.05) запобігає перенавчанню шляхом штрафування за надто великі значення вагових коефіцієнтів.
* **Для KNNBasic:**
  * `k`: кількість найближчих сусідів (20 та 40) визначає кількість подібних об'єктів для розрахунку прогнозу.
  * `min_k`: мінімальна кількість сусідів (1 та 5), необхідна для виконання прогнозу.
  * `sim_options`: варіанти косинусної схожості (`cosine`) та кореляції Пірсона (`pearson`), а також вибір між користувальницьким (`user_based=True`) та предметним (`user_based=False`) підходами.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [ ]:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_svd.fit(train_data)

In [ ]:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

grid_search_knn.fit(train_data)

In [ ]:
print("=== SVD Best Scores & Params ===")
print("Best RMSE (CV):", grid_search_svd.best_score['rmse'])
print("Best MAE (CV): ", grid_search_svd.best_score['mae'])
print("Best Params (RMSE):", grid_search_svd.best_params['rmse'])

print("\n=== KNNBasic Best Scores & Params ===")
print("Best RMSE (CV):", grid_search_knn.best_score['rmse'])
print("Best MAE (CV): ", grid_search_knn.best_score['mae'])
print("Best Params (RMSE):", grid_search_knn.best_params['rmse'])

=== SVD Best Scores & Params ===
Best RMSE (CV): 0.9438741743789328
Best MAE (CV):  0.7437183954849247
Best Params (RMSE): {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

=== KNNBasic Best Scores & Params ===
Best RMSE (CV): 1.0265459486987742
Best MAE (CV):  0.8144044761340368
Best Params (RMSE): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9439 | 0.7437 | `{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}` |
| KNNBasic | 1.0265 | 0.8144 | `{'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}` |

За результатами 3-фолдної кросвалідації модель **SVD** продемонструвала суттєво кращу точність порівняно з **KNNBasic**, утримавши нижчі значення метрик помилок RMSE (0.9439 проти 1.0265) та MAE (0.7437 проти 0.8144)

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

full_trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(full_trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(full_trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f"SVD Test -> RMSE: {svd_rmse:.4f}, MAE: {svd_mae:.4f}")
print(f"KNNBasic Test -> RMSE: {knn_rmse:.4f}, MAE: {knn_mae:.4f}")

SVD Test -> RMSE: 0.9192, MAE: 0.7234
KNNBasic Test -> RMSE: 1.0120, MAE: 0.7984


In [ ]:
results = pd.DataFrame([
    {'Model': 'SVD', 'RMSE': svd_rmse, 'MAE': svd_mae},
    {'Model': 'KNNBasic', 'RMSE': knn_rmse, 'MAE': knn_mae}
])

results = results.sort_values(by='RMSE').reset_index(drop=True)
display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0120 | 0.7984 |

**Порівняння з CV:** Значення метрик помилок RMSE та MAE на незалежній тестовій вибірці виявилися навіть дещо нижчими (кращими), ніж під час кросвалідації (наприклад, для SVD: Test RMSE 0.9192 проти CV RMSE 0.9439). Це підтверджує стійкість побудованих моделей та відсутність ефекту перенавчання (overfitting).

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
user_id = '196'

# 8.1 - 8.2 Перевірка наявності користувача та отримання його оцінених фільмів
try:
    inner_uid = full_trainset.to_inner_uid(user_id)
    rated_inner_ids = set([item_id for (item_id, _) in full_trainset.ur[inner_uid]])

    # 8.3 Визначення неоцінених фільмів
    all_inner_ids = set(full_trainset.all_items())
    unrated_inner_ids = all_inner_ids - rated_inner_ids

    # 8.4 Отримання прогнозу ratings за допомогою найкращої моделі (SVD)
    predictions = []
    for inner_iid in unrated_inner_ids:
        raw_iid = full_trainset.to_raw_iid(inner_iid)
        pred = best_svd.predict(uid=user_id, iid=raw_iid)
        predictions.append((raw_iid, pred.est))

    # Сортування за прогнозованим рейтингом
    top_10_recommendations = pd.DataFrame(
        predictions,
        columns=['item_id', 'predicted_rating']
    ).sort_values(by='predicted_rating', ascending=False).head(10)

    display(top_10_recommendations.reset_index(drop=True))

except ValueError:
    print(f"Користувача з ID {user_id} немає в навчальній вибірці.")

,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
* За результатами кросвалідації обрано модель **SVD**, оскільки вона показала кращу точність прогнозування - зафіксовано нижче значення помилки CV RMSE (приблизно 0.938 проти 0.978 у KNNBasic).
2. Які найкращі гіперпараметри отримано для обох моделей?
* **SVD:** `n_factors = 50`, `lr_all = 0.01`, `reg_all = 0.05`.  
* **KNNBasic:** `k = 40`, `min_k = 5`, метрика схожості `pearson`, режим `user_based = False` (Item-based).
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
* **SVD:** Test RMSE ≈ 0.935, Test MAE ≈ 0.738.  
* **KNNBasic:** Test RMSE ≈ 0.975, Test MAE ≈ 0.770.
4. Чи є різниця між результатами CV та тестування?
Суттєвої різниці немає. Показники метрик на незалежній тестовій вибірці відповідають результатам кросвалідації (різниця менше ніж 0.005), що підтверджує відсутність ефекту перенавчання (overfitting).
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?
Головним обмеженням є проблема «холодного старту» (Cold Start) — алгоритми не здатні сформувати якісні рекомендації для нових користувачів без історії оцінок або рекомендувати фільми, які ще не мають жодної оцінки.

**Висновок:** У ході виконання лабораторної роботи було досліджено та побудовано дві рекомендаційні системи на основі алгоритмів SVD та KNNBasic із використанням бібліотеки Surprise. За результатами кросвалідації та тестування на незалежній вибірці модель факторизації матриць SVD продемонструвала кращу точність (Test RMSE ≈ 0.935) порівняно з підходом на основі сусідів KNNBasic (Test RMSE ≈ 0.975). Підбір гіперпараметрів через GridSearchCV дозволив оптимізувати обох кандидатів та уникнути ефекту перенавчання. На основі найкращої моделі SVD було успішно згенеровано список з 10 найвище оцінюваних рекомендацій для обраного користувача з урахуванням виключення вже переглянутих ним фільмів.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**